# Hiyoung PPE YOLO Training In VSCode Colab

이 노트북은 VSCode Colab 확장에서 실행합니다.

- 먼저 VSCode에서 Colab 런타임에 연결해야 합니다.
- Google Drive에 `AI_Learn` 폴더가 업로드 또는 동기화되어 있어야 합니다.
- 로컬 Desktop 경로가 아니라 `/content/drive/MyDrive/AI_Learn` 기준으로 실행합니다.
- 최종 클래스 이름은 반드시 `helmet`, `person`, `vest` 순서를 유지해야 합니다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/AI_Learn"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())
print("exists:", os.path.exists(PROJECT_ROOT))
print("datasets exists:", os.path.exists("datasets/hiyoung_ppe"))
print("scripts exists:", os.path.exists("scripts"))
print("notebooks exists:", os.path.exists("notebooks"))

In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -U ultralytics

In [ ]:
!python scripts/check_yolo_dataset.py --yaml datasets/hiyoung_ppe_colab.yaml

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AI_Learn")
LOCAL_ROOT = Path("/content/AI_Learn")

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

# scripts, datasets, weights만 복사
shutil.copytree(DRIVE_ROOT / "scripts", LOCAL_ROOT / "scripts")
shutil.copytree(DRIVE_ROOT / "datasets", LOCAL_ROOT / "datasets")
shutil.copytree(DRIVE_ROOT / "weights", LOCAL_ROOT / "weights")

os.chdir(LOCAL_ROOT)
print("cwd:", os.getcwd())
print("dataset exists:", (LOCAL_ROOT / "datasets/hiyoung_ppe").exists())

In [ ]:
!python scripts/train_yolo.py \
  --model yolo11s.pt \
  --epochs 10 \
  --imgsz 640 \
  --data datasets/hiyoung_ppe_colab.yaml \
  --name test_helmet_person_vest_yolo11s

In [ ]:
!python scripts/validate_model.py \
  --weights runs/test_helmet_person_vest_yolo11s/weights/best.pt

## Full Training

테스트 학습과 검증이 성공한 뒤에만 아래 본 학습 셀을 실행합니다.
시간과 GPU 사용량이 크므로 기본적으로 바로 실행하지 않는 것을 권장합니다.

In [ ]:
!python scripts/train_yolo.py \
  --model yolo11m.pt \
  --epochs 120 \
  --imgsz 960 \
  --data datasets/hiyoung_ppe_colab.yaml \
  --name helmet_person_vest_yolo11m_960

In [ ]:
!python scripts/validate_model.py \
  --weights runs/helmet_person_vest_yolo11m_960/weights/best.pt

!python scripts/export_weight_to_project.py \
  --source runs/helmet_person_vest_yolo11m_960/weights/best.pt \
  --target-dir weights \
  --target-name hiyoung_helmet_person_vest_yolo11m.pt